# Module 0 — Foundations: the data and the toolbox

**Goal:** connect to the OSRS price store and load data into pandas comfortably.

This notebook is the smoke test for the whole curriculum. By the end you should be able to:

1. Confirm the InfluxDB 3 connection works.
2. Pull a single item's price series into a tidy, time-indexed DataFrame.
3. Downsample from raw 5-minute data to hourly/daily.
4. Load several items at once and reshape them.
5. Handle the fact that low-volume items don't trade every window.

> **Before running:** copy `.env.example` to `.env` at the repo root and fill in the
> InfluxDB 3 connection values (see `.kiro/steering/data-access.md`), then
> `pip install -r requirements.txt` and select that environment as the kernel.

## 1. Setup

The shared helper `ge_data.py` (in the `curriculum/` folder) reads the connection
settings from `.env` and exposes `load_series`, `load_frame`, and `list_items`. We add
the parent `curriculum/` folder to the path so the helper imports cleanly from this
sub-folder.

In [ ]:
import sys
from pathlib import Path

# Make the shared curriculum helpers importable from this sub-folder.
CURRICULUM_DIR = Path.cwd().parent
if str(CURRICULUM_DIR) not in sys.path:
    sys.path.insert(0, str(CURRICULUM_DIR))

import pandas as pd
import matplotlib.pyplot as plt

import ge_data
from ge_data import load_series, load_frame, list_items, FIRE_RUNE

pd.set_option("display.max_columns", None)
print("pandas", pd.__version__)

## 2. Confirm the connection

`ge_data.get_client()` raises a clear error if the `.env` variables are missing. A tiny
bounded query against the reference item (Fire rune, `554`) confirms the store is
reachable and returning data.

In [ ]:
# A high-volume item that trades in essentially every 5-minute window -> good smoke test.
smoke = load_series(FIRE_RUNE, start="2023-01-01", stop="2023-01-02", interval="1h")
print(f"Rows returned: {len(smoke)}")
smoke.head()

If the cell above returns rows, the connection works.

> **No data?** The live store is loaded oldest-first and may not have reached recent
> dates yet. Try an older window (e.g. 2021-2022) or widen the range. Always keep an
> `interval` on ranges longer than a few days so the query stays under the store's
> file-scan limit.

## 3. Load and plot a year of Fire rune prices

The **key concept** here is downsampling: the raw data is one point every 5 minutes, but
for a year-long view we bin it hourly (`interval="1h"`). Each bin is the mean over that
hour. The helper returns a DataFrame indexed by a UTC `time`.

In [ ]:
# Parameterized at the top so this notebook is easy to re-point at another item/window.
ITEM_ID = FIRE_RUNE
START = "2023-01-01"
STOP = "2024-01-01"
INTERVAL = "1h"

series = load_series(ITEM_ID, start=START, stop=STOP, interval=INTERVAL)
print(f"{len(series):,} rows from {series.index.min()} to {series.index.max()}")
series.describe()

In [ ]:
fig, (ax_price, ax_vol) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

series[["avgHighPrice", "avgLowPrice"]].plot(ax=ax_price)
ax_price.set_ylabel("price (gp)")
ax_price.set_title(f"Item {ITEM_ID}: hourly avg high/low price")

series[["highPriceVolume", "lowPriceVolume"]].plot(ax=ax_vol)
ax_vol.set_ylabel("volume")
ax_vol.set_xlabel("time (UTC)")

plt.tight_layout()
plt.show()

**Interpretation (in market terms):** the high/low prices track the buy/sell sides of the
Grand Exchange; the gap between them is roughly the margin a flipper could capture, and
the volume panel shows how much actually traded. Spikes in one without the other are the
kind of thing later modules (outlier detection, volatility) will formalize.

## 4. Resolution matters: raw vs. hourly vs. daily

Downsampling trades detail for a longer, cheaper view. Compare a single week at three
resolutions. Keep raw (`interval=None`) only for short windows and methods that genuinely
need 5-minute resolution.

In [ ]:
week_start, week_stop = "2023-06-01", "2023-06-08"
raw = load_series(ITEM_ID, week_start, week_stop, interval=None)
hourly = load_series(ITEM_ID, week_start, week_stop, interval="1h")
daily = load_series(ITEM_ID, week_start, week_stop, interval="1d")

for name, df in [("raw 5m", raw), ("1h", hourly), ("1d", daily)]:
    print(f"{name:>7}: {len(df):>5} rows")

ax = raw["avgHighPrice"].plot(figsize=(12, 4), alpha=0.4, label="raw 5m")
hourly["avgHighPrice"].plot(ax=ax, label="1h mean")
daily["avgHighPrice"].plot(ax=ax, marker="o", label="1d mean")
ax.legend()
ax.set_title(f"Item {ITEM_ID}: same week at three resolutions")
plt.show()

## 5. Several items at once

`load_frame` returns a long-form DataFrame (`itemID`, `time`, + fields). Pivoting to wide
gives one column per item, which is the shape most analysis and modeling wants. Here we
compare a couple of runes.

In [ ]:
# 554 = Fire rune, 555 = Water rune, 556 = Air rune, 557 = Earth rune.
items = ["554", "555", "556", "557"]
long = load_frame(items, start="2023-06-01", stop="2023-07-01", interval="1h")
long.head()

In [ ]:
wide_high = long.pivot_table(index="time", columns="itemID", values="avgHighPrice")
wide_high.plot(figsize=(12, 4), title="Hourly avg high price by item")
plt.ylabel("price (gp)")
plt.show()
wide_high.tail()

## 6. Missing windows

Low-volume items don't trade in every 5-minute window, so their series have gaps. Detect
them by reindexing onto a complete time grid: any `NaN` is a window with no trade. How
you fill these (forward-fill, interpolate, or leave as missing) is a modeling decision we
revisit in the time-series modules.

In [ ]:
one_week = load_series(ITEM_ID, week_start, week_stop, interval="1h")
full_grid = pd.date_range(one_week.index.min(), one_week.index.max(), freq="1h")
reindexed = one_week.reindex(full_grid)

missing = int(reindexed["avgHighPrice"].isna().sum())
print(f"{missing} of {len(full_grid)} hourly windows have no avgHighPrice")
reindexed["avgHighPrice"].isna().mean()

## Recap

You connected to the store, loaded single- and multi-item series, downsampled across
resolutions, and saw where data is missing. Everything after this builds on these loads.

**Next:** Module 1 — Exploratory data analysis & descriptive statistics.